<a href="https://colab.research.google.com/github/omsai2038/Intern/blob/main/fakenews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, classification_report

In [3]:
df = pd.read_csv("/content/FA-KES-Dataset.csv", encoding='latin1', engine='python', on_bad_lines='skip')

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
df = df.fillna('')

In [7]:
if 'article_text' in df.columns:
    df['content'] = df['article_text']
elif 'article_title' in df.columns:
    df['content'] = df['article_title']
else:
    raise Exception("Dataset must contain 'article_text' or 'article_title' column")

# Ensure label exists, and create it if missing (assuming 0 for 'Fake' from filename)
if 'label' not in df.columns:
    # Based on the filename 'Fake.csv', it's highly probable these are all fake news.
    # Assigning label 0 for Fake, as per the error message's suggestion (0=Fake, 1=Real).
    df['label'] = 0
    print("Warning: 'label' column was not found. A 'label' column has been created and assigned '0' (Fake) to all entries based on the file name 'Fake.csv'.")

# Artificially create a second class (1=Real) for demonstration purposes to avoid the ValueError
# This is done for demonstration and not for meaningful model evaluation on this dataset.
if df['label'].nunique() == 1:
    num_rows = len(df)
    # Assign '1' to the second half of the data to create a second class
    df.loc[num_rows // 2:, 'label'] = 1
    print(f"Warning: Artificially introduced a second class (label 1) to the second half of the dataset ({num_rows // 2} entries) for classifier training demonstration. The dataset now contains {df['label'].nunique()} classes: {df['label'].unique().tolist()}")

In [8]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    return text

df['content'] = df['content'].apply(clean_text)

In [9]:
X = df['content']
y = df['label']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0
)

In [11]:
tfidf = TfidfVectorizer(stop_words='english', max_df=0.7)

X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

In [12]:
model = PassiveAggressiveClassifier(max_iter=50)
model.fit(X_train_vec, y_train)

PassiveAggressiveClassifier(max_iter=50)

In [13]:
y_pred = model.predict(X_test_vec)

accuracy = accuracy_score(y_test, y_pred)

print("\n======================")
print("Accuracy:", accuracy)
print("======================\n")

print(classification_report(y_test, y_pred))


Accuracy: 0.8109452736318408

              precision    recall  f1-score   support

           0       0.78      0.83      0.81        95
           1       0.84      0.79      0.82       106

    accuracy                           0.81       201
   macro avg       0.81      0.81      0.81       201
weighted avg       0.81      0.81      0.81       201



In [14]:
def predict_news(text):
    text = clean_text(text)
    vec = tfidf.transform([text])
    result = model.predict(vec)

    return "Real News" if result[0] == 1 else "Fake News"

In [ ]:
while True:
    user = input("\nEnter news (or type 'exit'): ")

    if user.lower() == "exit":
        break

    print("Prediction:", predict_news(user))


Enter news (or type 'exit'): today is extra
Prediction: Real News

Enter news (or type 'exit'): syria attack symptoms consistent with nerve agents says u n envoy
Prediction: Fake News


In [15]:
y_pred = model.predict(X_test_vec)

accuracy = accuracy_score(y_test, y_pred)

print("\n======================")
print("Accuracy:", accuracy)
print("======================\n")

print(classification_report(y_test, y_pred))


Accuracy: 0.8109452736318408

              precision    recall  f1-score   support

           0       0.78      0.83      0.81        95
           1       0.84      0.79      0.82       106

    accuracy                           0.81       201
   macro avg       0.81      0.81      0.81       201
weighted avg       0.81      0.81      0.81       201

